In [1]:
with open("/content/drive/MyDrive/nutuk.txt") as f:
    text = f.read()


In [3]:
print("length of the characters: ", len(text))

length of the characters:  1577732


In [4]:
print(text[9000:10000])

sında
Diyarbekir (Vesika: 8, 9), Bitlis, Elaziz vilayetlerin­
de, İstanbul'dan idare olunan Kürt Teafi Cemiyeti!
vardı. Bu cemiyetin maksadı, yabancı himayesi altında bir Kürt hükümeti vü­
cuda getirmekti.
Konya ve havalisinde, İstanbul'dan idare olunan Teaiii İslam Cemiyeti teş­
kiline çalışılıyordu. Memleketin hemen her tarafında İtilaf ve Hürriyet, Sulh
ve Selamet cemiyetleri de vardı.

Memleket dahilinde
ve İstanbul'da milli
varlığa düşman
teşekküller

İngiliz Muhipleri
Cemiyeti

İstanbul'da, muhtelif maksatlarla gizli ve açık olmak
üzere de birtakım fırka veya cemiyet unvanı altında te­
şekküller vardı.
İstanbul'da mühim sayılacak teşebbüslerden biri İngiliz Muhipleri Cemiyeti
idi. Bu isimden, İngilizlere muhip2 olanların teşkil ettiği bir cemiyet anlaşılma­
sın! Bence, bu cemiyeti teşkil edenler, kendi şahıslannı ve şahsi menfaatlannı
sevenler ve şahıslanyla menfaatlannın dokunulmazlığı çaresini Loyd Core3 hü­
kümeti marifetiyle İngiliz himayesini teminde arayanlardır. Bu bedbaht

In [6]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("total chars: ", vocab_size)
print(''.join(chars))
print(vocab_size)

total chars:  103

 !"%'()*,-./0123456789:;<>?ABCDEFGHIJKLMNOPQRSTUVWYZ[\]_abcdefghijklmnopqrstuvwxyz{§«­·ÇÖÜçôöüğİıŞş•�
103


In [8]:
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s] #encoder: take a string,output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) #decoder: take a list of integers, output a string

print(encode('noktali virgul'))
print(decode(encode('noktali virgul')))

[71, 72, 68, 77, 58, 69, 66, 2, 79, 66, 75, 64, 78, 69]
noktali virgul


In [9]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[9000:10000])

torch.Size([1577732]) torch.int64
tensor([ 76,  98,  71,  61,  58,   0,  32,  66,  82,  58,  75,  59,  62,  68,
         66,  75,   2,   7,  50,  62,  76,  66,  68,  58,  24,   2,  22,  10,
          2,  23,   8,  10,   2,  30,  66,  77,  69,  66,  76,  10,   2,  33,
         69,  58,  83,  66,  83,   2,  79,  66,  69,  58,  82,  62,  77,  69,
         62,  75,  66,  71,  87,   0,  61,  62,  10,   2,  97,  76,  77,  58,
         71,  59,  78,  69,   6,  61,  58,  71,   2,  66,  61,  58,  75,  62,
          2,  72,  69,  78,  71,  58,  71,   2,  39,  95,  75,  77,   2,  48,
         62,  58,  63,  66,   2,  31,  62,  70,  66,  82,  62,  77,  66,   3,
          0,  79,  58,  75,  61,  98,  12,   2,  30,  78,   2,  60,  62,  70,
         66,  82,  62,  77,  66,  71,   2,  70,  58,  68,  76,  58,  61,  98,
         10,   2,  82,  58,  59,  58,  71,  60,  98,   2,  65,  66,  70,  58,
         82,  62,  76,  66,   2,  58,  69,  77,  98,  71,  61,  58,   2,  59,
         66,  75,   2,  39,  9

In [11]:
#split
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [15]:
block_size = 8
train_data[:block_size+1], train_data[:block_size*2]

(tensor([42, 49, 48, 49, 39, 37,  0,  0,  7]),
 tensor([42, 49, 48, 49, 39, 37,  0,  0,  7, 15, 23, 15, 23, 11, 15, 23]))

In [17]:
x = train_data[:block_size*2]
y = train_data[1:block_size*2+1]
for t in range(block_size):
  context = x[t:block_size+t+1]
  target = y[block_size+t]
  print(f"when input is {context} the target: {target}")

when input is tensor([42, 49, 48, 49, 39, 37,  0,  0,  7]) the target: 15
when input is tensor([49, 48, 49, 39, 37,  0,  0,  7, 15]) the target: 23
when input is tensor([48, 49, 39, 37,  0,  0,  7, 15, 23]) the target: 15
when input is tensor([49, 39, 37,  0,  0,  7, 15, 23, 15]) the target: 23
when input is tensor([39, 37,  0,  0,  7, 15, 23, 15, 23]) the target: 11
when input is tensor([37,  0,  0,  7, 15, 23, 15, 23, 11]) the target: 15
when input is tensor([ 0,  0,  7, 15, 23, 15, 23, 11, 15]) the target: 23
when input is tensor([ 0,  7, 15, 23, 15, 23, 11, 15, 23]) the target: 16


In [20]:
torch.manual_seed(42)
batch_size = 4
block_size = 8

def get_batch(split):
  data = train_data if split == 'train' else val_data
  ix = torch.randint(len(data) - block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('-----')

for b in range(batch_size):
  for t in range(block_size):
    context = xb[b, :t+1]
    target = yb[b,t]
    print(f"when input is {context.tolist()} the target: {target}")


inputs:
torch.Size([4, 8])
tensor([[ 12,   2,  30,  58,  75,  98, 100,   2],
        [ 58,  69,  98, 100,  98,  69,  70,  98],
        [ 58,  82,  66,  70,   2,  79,  62,   2],
        [ 70,  98, 100,  77,  98,  75,  12,   2]])
targets:
torch.Size([4, 8])
tensor([[  2,  30,  58,  75,  98, 100,   2,  68],
        [ 69,  98, 100,  98,  69,  70,  98, 100],
        [ 82,  66,  70,   2,  79,  62,   2,  71],
        [ 98, 100,  77,  98,  75,  12,   2,  97]])
-----
when input is [12] the target: 2
when input is [12, 2] the target: 30
when input is [12, 2, 30] the target: 58
when input is [12, 2, 30, 58] the target: 75
when input is [12, 2, 30, 58, 75] the target: 98
when input is [12, 2, 30, 58, 75, 98] the target: 100
when input is [12, 2, 30, 58, 75, 98, 100] the target: 2
when input is [12, 2, 30, 58, 75, 98, 100, 2] the target: 68
when input is [58] the target: 69
when input is [58, 69] the target: 98
when input is [58, 69, 98] the target: 100
when input is [58, 69, 98, 100] the target: 9

# Token Embeddibng:

In [23]:
def _init_ (self):
  super()._init_()
  self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
  self.position_embedding_table = nn.Embedding(block_size, n_embd)
  self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
  sel.ln_f = nn.LayerNorm(n_embd) # final layer norm
  self.lm_head = nn.Linear(n_embd, vocab_size)

  self.apply(self._init_weights)

# Weighted Aggregation In Self Attention:

In [24]:
torch.manual_seed(1337)
B,T,C = 4,8,2
x = torch.randn(B,T,C)
print(x.shape)
print(x[0])

torch.Size([4, 8, 2])
tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])


In [25]:
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow = wei @ x
print(wei)
print(x[0])
print(xbow[0])

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])
tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])
tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.09

## Self Attention

In [27]:
# key: what we have
# query: what we looking for
# value = x değeri
# wei = query*key
# wei*x


In [29]:
torch.manual_seed(1337)
B, T, C = 4,8,32
x = torch.randn(B,T,C)

head_size = 16
key  = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)
q = query(x)
wei = q @ k.transpose(-2, -1)


tril = torch.tril(torch.ones(T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

tril = torch.tril(torch.ones(T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)


v = value(x)
out = wei @ v

print(out.shape)
print(out[0])

torch.Size([4, 8, 16])
tensor([[-0.1571,  0.8801,  0.1615, -0.7824, -0.1429,  0.7468,  0.1007, -0.5239,
         -0.8873,  0.1907,  0.1762, -0.5943, -0.4812, -0.4860,  0.2862,  0.5710],
        [ 0.5006, -0.2466, -0.1615,  0.0830, -0.1311, -0.0755, -0.3178, -0.1964,
         -0.2260,  0.6137,  0.5997, -0.2172,  0.1563,  0.1683,  0.0044,  1.1238],
        [ 0.4476, -0.0802, -0.3119,  0.0955,  0.0699, -0.0908, -0.0592, -0.0645,
         -0.2826, -0.1032,  0.3499,  0.0248, -0.1936, -0.0363, -0.0802,  1.1567],
        [ 0.4068, -0.0845, -0.2871,  0.0318,  0.1930, -0.1398, -0.0345, -0.0953,
         -0.1938,  0.0910,  0.1266,  0.0054, -0.0561,  0.0650,  0.1895,  0.8319],
        [ 0.3765,  0.2044,  0.0621,  0.1823,  0.2566,  0.1835,  0.2193,  0.1309,
         -0.3135, -0.4260, -0.0834, -0.0718, -0.3925,  0.1358,  0.0538,  0.7487],
        [ 0.2218,  0.1336, -0.0230,  0.2807,  0.2616,  0.1399,  0.0904,  0.0177,
         -0.1801, -0.2140, -0.0273,  0.0754, -0.2312,  0.1449,  0.1918,  0.6403],

In [30]:
n_emb = 256
dropout = 0.2

class Head(nn.Module):
  """ one head self attention """

  def __init__(self, head_size):
    super().__init__()
    key  = nn.Linear(C, head_size, bias=False)
    query = nn.Linear(C, head_size, bias=False)
    value = nn.Linear(C, head_size, bias=False)
    self.register_buffer('tril', torch.tril(torch.ones(block_size,block_size)))

    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    B,T,C = x.shape
    k = self.key(x)
    q = self.query(x)

    wei = q @ k.transpose(-2,1) * k.shape[-1]**-0.5
    wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
    wei = F.softmax(wei, dim=-1)
    wei = self.dropout(wei)

    v = self.value(x)
    out = wei @ v
    return out

In [31]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, head_size):
      super()._init_()
      self.heads = nn.ModuleList((Head(head_size) for _ in range(num_heads)))
      self.proj = nn.Linear(head_size * num_heads, n_embd)
      self.dropout = nn.Dropout(dropout)

    def forward(self, x):
      out = torch.cat([h(x) for h in self.heads], dim=-1)
      out = self.dropout(self.proj(out))
      return out

In [32]:
# feed forward

class FeedForward(nn.Module):
  def __init__(self, n_embd):
    super()._init_()
    self.net = nn.Sequential(
        nn.Linear(n_embd, 4 * n_embd),
        nn.ReLU(),
        nn.Linear(4 * n_embd, n_embd),
        nn.Dropout(dropout),
    )
  def forward(self, x):
    return self.net(x)

In [33]:
class Block(nn.Module):

  def _init_(self, n_embd, n_head):
    super()._init_()
    head_size = n_embd // n_head
    self.sa = MultiHeadAttention(n_head, head_size)
    self.ffwd

    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self, x):
    x = x + self.sa(self.ln1(x)) #residual connection
    x = x + self.sa(self.ln2(x))
    return x


In [37]:
n_head = 6
n_layer = 6

class GPTLanguageModel(nn.Module):

  def _init_ (self):
    super()._init_()
    self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
    self.position_embedding_table = nn.Embedding(block_size, n_embd)
    self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
    sel.ln_f = nn.LayerNorm(n_embd) # final layer norm
    self.lm_head = nn.Linear(n_embd, vocab_size)

    self.apply(self._init_weights)

  def _init_weights(self, module):
    if isinstance(module, nn.Linear):
      torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
      if module.bias is not None:
        torch.nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
      torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

  def forward(self, idx, target=None):
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if target is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view
            target = target.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

  def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):

            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        return idx


In [ ]:
#train

import torch
import torch.nnn as nn
form torch.nn import functional as F

#hyperparameters
batch_size = 64
blocks_ size = 256
max_iters = 5000
eval_interval = 500
eval_iters = 50
learning_rate = 3e-4
deviced = 'cuda' if torch.cuda.is_available() else 'cpu'
n_embd = 256
n_head = 6
n_layer = 6
dropout = 0.2


torch.manual_seed(1337)

with open("/content/drive/MyDrive/nutuk.txt") as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)


stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s] #encoder: take a string,output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) #decoder: take a list of integers, output a string

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
  data = train_data if split == 'train' else val_data
  ix = torch.randint(len(data) - block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  return x, y


  @torch.no_grad()

